In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install --upgrade transformers accelerate torch torchvision timm nltk --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.8 MB/s eta 0:00:00


In [4]:
from huggingface_hub import login
login(new_session=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
from transformers import pipeline

pipe = pipeline("image-feature-extraction", model="facebook/dinov3-vith16plus-pretrain-lvd1689m")

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.36G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

Device set to use cuda:0


In [6]:
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained("facebook/dinov3-vith16plus-pretrain-lvd1689m")
model = AutoModel.from_pretrained("facebook/dinov3-vith16plus-pretrain-lvd1689m")

In [9]:
import os
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import timm
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoImageProcessor, AutoModel

print("Loading DINOv3 visual encoder...")
processor = AutoImageProcessor.from_pretrained("facebook/dinov3-vith16plus-pretrain-lvd1689m")
vision_model = AutoModel.from_pretrained("facebook/dinov3-vith16plus-pretrain-lvd1689m").to("cuda")
vision_model.eval()

Loading DINOv3 visual encoder...


DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 1280, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (layer): ModuleList(
    (0-31): 32 x DINOv3ViTLayer(
      (norm1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (attention): DINOv3ViTAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (o_proj): Linear(in_features=1280, out_features=1280, bias=True)
      )
      (layer_scale1): DINOv3ViTLayerScale()
      (drop_path): Identity()
      (norm2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (mlp): DINOv3ViTGatedMLP(
        (gate_proj): Linear(in_features=1280, out_features=5120, bias=True)
        (up_proj): Linear(in_features=1280, out_features=5120, bias=True)
        (down_proj

In [10]:
def extract_features(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = vision_model(**inputs)

        # DINOv3 stores global CLS embeddings in `last_hidden_state[:, 0, :]`
        if hasattr(outputs, "last_hidden_state"):
            embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        else:
            raise ValueError("Unexpected output structure from DINOv3")

    return embedding.squeeze().cpu()

In [11]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
text_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


... (4 outputs left out)

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [12]:
test_folder = "/content/drive/MyDrive/VLM_Project/VQA-Med-2019/VQAMed2019Test"
questions_file = os.path.join(test_folder, "VQAMed2019_Test_Questions_w_Ref_Answers.txt")

df = pd.read_csv(
    questions_file,
    sep="|",
    header=None,
    names=["image_id", "category", "question", "answer"]
)
images_folder = os.path.join(test_folder, "VQAMed2019_Test_Images")
print("Total questions:", len(df))

Total questions: 500


In [13]:
def run_vqa(image_path, question, max_new_tokens=80):
    emb = extract_features(image_path)
    emb_str = " ".join([f"{x:.4f}" for x in emb[:32]])

    messages = [
        {"role": "system",
         "content": "You are a helpful medical assistant answering questions about medical images."},
        {"role": "user",
         "content": f"The image embedding is: {emb_str}\n\nQuestion: {question}\nPlease provide a short, medically relevant answer."}
    ]

    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = text_model.generate(**inputs, max_new_tokens=max_new_tokens)

    answer = tokenizer.decode(output[0], skip_special_tokens=True)

    for key in ["assistant", "Assistant", "Answer:", "answer:"]:
        if key in answer:
            answer = answer.split(key)[-1].strip()

    return answer


In [14]:
img_path = os.path.join(images_folder, df.loc[0, "image_id"] + ".jpg")
question = df.loc[0, "question"]
ans = run_vqa(img_path, question)
print("Q:", question)
print("A:", ans)

Q: what modality is shown?
A: Based on the vector provided, it appears to represent a modality-agnostic image embedding, which could be from various imaging modalities such as MRI, CT, or X-ray. Without additional context, it's not possible to determine the specific modality.


In [15]:
results = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    img_path = os.path.join(images_folder, row["image_id"] + ".jpg")
    question = row["question"]
    gt_answer = str(row["answer"]).strip().lower()
    try:
        pred_answer = run_vqa(img_path, question)
    except Exception as e:
        pred_answer = f"Error: {e}"

    results.append({
        "image_id": row["image_id"],
        "question": question,
        "gt_answer": gt_answer,
        "pred_answer": pred_answer
    })

results_df = pd.DataFrame(results)
results_df.to_csv("dinov2_qwen_baseline_predictions.csv", index=False)
print("Saved predictions to dinov2_qwen_baseline_predictions.csv")

100%|██████████| 500/500 [26:39<00:00,  3.20s/it]

Saved predictions to dinov2_qwen_baseline_predictions.csv


In [16]:
def exact_match(pred, gt):
    return int(pred.strip().lower() == gt.strip().lower())

results_df["exact_match"] = [
    exact_match(p, g) for p, g in zip(results_df["pred_answer"], results_df["gt_answer"])
]
accuracy = results_df["exact_match"].mean()
print(f"Exact Match Accuracy: {accuracy:.4f}")

from difflib import SequenceMatcher
def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

results_df["similarity"] = [
    similarity(p, g) for p, g in zip(results_df["pred_answer"], results_df["gt_answer"])
]
mean_sim = results_df["similarity"].mean()
print(f"Semantic Similarity: {mean_sim:.4f}")

Exact Match Accuracy: 0.0000
Semantic Similarity: 0.0548


In [17]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt')

def compute_bleu(pred, gt):
    smoothie = SmoothingFunction().method4
    return sentence_bleu([gt.lower().split()], pred.lower().split(), smoothing_function=smoothie)

results_df["bleu"] = [compute_bleu(p, g) for p, g in zip(results_df["pred_answer"], results_df["gt_answer"])]
mean_bleu = results_df["bleu"].mean()
print(f"BLEU Score: {mean_bleu:.4f}")

summary = {
    "Model": "DINOv2 + Qwen2.5-7B-Instruct",
    "Num_Questions": len(df),
    "Exact_Match_Accuracy": accuracy,
    "Mean_Similarity": mean_sim,
    "Mean_BLEU": mean_bleu
}
pd.DataFrame([summary]).to_csv("dinov2_qwen_baseline_summary.csv", index=False)
print("Saved summary.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


BLEU Score: 0.0013
Saved summary.
